# dl_02_extract_qbo

Pulls the entities in `Files/config/qbo_entities.yml` into bronze.

**The refresh token rotates on every use** and hard-expires at 100 days.
The new one is persisted to `dl_meta_token` immediately after the
exchange - before any data is pulled - because a crash mid-run must not
lose the only credential that still works.

In [ ]:
import sys
sys.path.insert(0, "/lakehouse/default/Files/lib")

LIB = "/lakehouse/default/Files"

import yaml
import requests

import fabric_common as fc
import qbo_extract as qx
import watermark as wm
from ratelimit import RateLimitedSession

CONFIG = f"{LIB}/config/qbo_entities.yml"
batch_id = fc.new_batch_id()

with open(CONFIG, encoding="utf-8") as handle:
    config = yaml.safe_load(handle)
entities = config["entities"]
reports = config.get("reports", [])
print(f"batch {batch_id}: {len(entities)} entities, {len(reports)} reports")

In [ ]:
settings = qx.settings_from_secrets(fc.get_secret)
session = RateLimitedSession(requests.Session(), header_units="seconds")

# The stored token beats the one in configuration. QBO invalidates the previous
# refresh token the moment a new one is issued, so the value in .env or Key
# Vault is stale after the very first successful run.
stored = wm.read_token(spark, "quickbooks")
refresh_token = stored[0] if stored else fc.get_secret("QUICKBOOKS_REFRESH_TOKEN")

tokens = qx.refresh_access_token(settings, session, refresh_token)

# PERSIST FIRST, PULL SECOND. If this write is skipped or fails, the rotated
# token is lost and the integration is dead until someone re-consents by hand.
wm.write_token(spark, "quickbooks", tokens.refresh_token, batch_id)
headers = qx.build_headers(tokens.access_token)
print(f"authenticated to realm {settings.realm_id} ({settings.environment})")

In [ ]:
from datetime import datetime, timedelta, timezone

summary = []

for entry in entities:
    name = entry["name"]
    table = entry["bronze_table"]
    full_reload = entry.get("full_reload", False)

    since = None
    where = None
    if not full_reload:
        since = wm.read_since(spark, table, name)
        if since:
            where = qx.changed_since_where(since)

    records = list(qx.iter_entity(session, settings, headers, name, where=where))
    ingested_at = fc.utc_now()
    rows = [
        {
            **qx.to_bronze_row(record, name, ingested_at),
            "_batch_id": batch_id,
            "_row_hash": fc.row_hash(record),
        }
        for record in records
    ]

    if rows:
        df = spark.createDataFrame(rows)
        fc.merge_delta(spark, df, table, ["_merge_key"])
        high = qx.high_water(records)
        if not full_reload and high:
            wm.write_watermark(spark, table, name, high, batch_id)

    fc.log_run(spark, batch_id, "extract_qbo", table, len(rows))
    mode = "full" if full_reload else ("incremental" if since else "initial")
    summary.append((name, len(rows), mode))
    print(f"  {name:24s} {len(rows):7,d} rows  ({mode})")

In [ ]:
import json

# Reports return a nested Rows/ColData tree rather than a list, so each is
# stored WHOLE and flattened in silver where the shape is visible in SQL.
for entry in reports:
    name = entry["name"]
    table = entry["bronze_table"]
    payload = qx.fetch_report(session, settings, headers, name, entry.get("params"))
    row = [{
        "_key": name,
        "_project_id": None,
        "_merge_key": f"report|{name}",
        "_source_endpoint": name,
        "_ingested_at": fc.utc_now(),
        "payload": json.dumps(payload, default=str),
        "_batch_id": batch_id,
        "_row_hash": fc.row_hash(payload),
    }]
    fc.merge_delta(spark, spark.createDataFrame(row), table, ["_merge_key"])
    fc.log_run(spark, batch_id, "extract_qbo_report", table, 1)
    print(f"  {name:24s} report captured")

In [ ]:
import json


def write_diag(name: str, payload: dict) -> None:
    """Structured diagnostics to Files/_diag/.

    Fabric's job API gives no per-cell detail - a failed notebook reports
    "Failed" and nothing else. Writing what happened to a file the deploy
    scripts can read back is the difference between debugging this and guessing.
    """
    os.makedirs("/lakehouse/default/Files/_diag", exist_ok=True)
    path = f"/lakehouse/default/Files/_diag/{name}.json"
    with open(path, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, default=str)
    print(f"diagnostics -> {path}")

write_diag("extract_qbo", {
    "batch_id": batch_id,
    "realm": settings.realm_id,
    "environment": settings.environment,
    "refresh_token_rotated": True,
    "entities": [{"name": n, "rows": c, "mode": m} for n, c, m in summary],
})